# Iniciando o Spark

In [1]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_telco") \
    .getOrCreate()

# Importando bibliotecas

In [ ]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [ ]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

# Buckets e nomes de saída
bucket_base = "base_telco"
bucket_trusted = f"s3://hackathon_2025/{PROCESS_DATE}/0003_trusted/{bucket_base}"
bucket_raw = f"s3://hackathon_2025/{PROCESS_DATE}/0002_raw/{bucket_base}"
bucket_control = f"s3://hackathon_2025/{PROCESS_DATE}/0005_control/{bucket_base}"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_trusted:", bucket_trusted)
print("bucket_raw:", bucket_raw)
print("bucket_control:", bucket_control)


# Leitura dos dados na camada Raw

In [ ]:
path_raw = os.path.join(bucket_raw, bucket_base)
df_raw = spark.read.parquet(path_raw)
df_raw.createOrReplaceTempView("raw_base_telco")

print(log(), "Registros na Raw:", df_raw.count())
df_raw.show(5, truncate=False)


+-----------+--------------+---------+---------+-------+-----+------------------+----------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+------+
|    NUM_CPF|       dt_Proc|SAFRA_ANO|SAFRA_MES|IsSetup|IsFPD|ProductDescription|ProductMigration|Var26|Var27|Var28|Var29|Var30|Var31|Var32|Var33|Var34|Var35|Var36|Var37|Var38|Var39|Var40|Var41|Var42|Var43|Var44|Var45|Var46|Var47|Var48|Var49|Var50|Var51|Var52|Var53|Var54|Var55|Var56|Var57|Var58|Var59|Var60|Var61|Var62|Var63|Var64|Var65|Var66|Var67|Var68|Var69|Var70|Var71|Var72| Var73|Var74|Var75|Var76|Var77|Var78|Var79|Var80|Var81|Var82|Var83|Var84|Var85|Var86|Var87|Var88|Var89|Va

# Processamento tipagem para camada Trusted

In [ ]:
df_trusted = spark.sql(f"""
    SELECT
        ref,
        ref_partition,
        '{dthproc}' AS ts_proc,
        '{dthproc}' AS ts_proc_partition,
        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        CAST(SAFRA AS INT) AS SAFRA,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 1, 4) AS INT) AS SAFRA_ANO,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 5, 2) AS INT) AS SAFRA_MES,
        CAST(FLAG_INSTALACAO AS BOOLEAN) AS IsSetup,
        CAST(FPD AS BOOLEAN) AS IsFPD,
        CAST(PROD AS STRING) AS ProductDescription,
        CAST(flag_mig2 AS STRING) AS ProductMigration,
        {",".join([f"CAST(var_{i} AS FLOAT) AS Var{i}" for i in range(26,94)])}
    FROM raw_base_telco
""")

df_trusted.createOrReplaceTempView("lake_base_telco")
df_trusted.cache()

print(log(), "Registros Trusted:", df_trusted.count())
#df_trusted.printSchema()
df_trusted.show(5, truncate=False)

+-----------+--------------+---------+---------+-------+-----+------------------+----------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+------+
|    NUM_CPF|       dt_Proc|SAFRA_ANO|SAFRA_MES|IsSetup|IsFPD|ProductDescription|ProductMigration|Var26|Var27|Var28|Var29|Var30|Var31|Var32|Var33|Var34|Var35|Var36|Var37|Var38|Var39|Var40|Var41|Var42|Var43|Var44|Var45|Var46|Var47|Var48|Var49|Var50|Var51|Var52|Var53|Var54|Var55|Var56|Var57|Var58|Var59|Var60|Var61|Var62|Var63|Var64|Var65|Var66|Var67|Var68|Var69|Var70|Var71|Var72| Var73|Var74|Var75|Var76|Var77|Var78|Var79|Var80|Var81|Var82|Var83|Var84|Var85|Var86|Var87|Var88|Var89|Va

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_trusted.write \
    .partitionBy("SAFRA,ref_partition","ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

# Controle de carga

In [ ]:
controle = spark.sql(f"""
    SELECT
        '{output_trusted}' AS name_file,
        ref,
        ref_partition,
        ts_proc,
        ts_proc_partition,
        COUNT(*) AS qtd_registros
    FROM lake_base_telco
    GROUP BY 1,2,3,4,5
""")

controle.createOrReplaceTempView("controle")
controle.cache()

print(log(), "Registros controle:", controle.count())
controle.show(truncate=False)

# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f"tb_0002_controle_processamento_{bucket_base}_trusted")
print("Control path:", path_control)

controle.write \
    .mode("append") \
    .option("compression", "snappy") \
    .parquet(path_control)